# Perbandingan scaler/transformer
**unsupervised › preprocessing** · notebook `scaling_compare`

**Kapan dipakai:** Model sensitif skala (KNN, SVM, logistik, neural) dan fitur punya skala/distribusi berbeda.

**Jebakan umum:** Scaler HARUS dilatih hanya pada data training (di dalam Pipeline). Model pohon tidak butuh scaling.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/unsupervised/wine.csv"; LABEL_COL = "cultivar"
MODEL_KEY = "scaling_compare"
OUT = out_dir("unsupervised", "preprocessing", MODEL_KEY)
SEED = 42; set_seed(SEED)

## 2. Data: skala fitur sangat berbeda (mis. `proline` ribuan vs `hue` ~1)

In [ ]:
df = pd.read_csv(PATH); y = df[LABEL_COL].to_numpy(); X = df.drop(columns=[LABEL_COL]).select_dtypes("number")
display(X.describe().loc[["mean", "std", "min", "max"]].round(2).T.head(8))

## 3. Scaler × model, dengan scaler di DALAM pipeline (belajar hanya dari data training)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer, QuantileTransformer, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
scalers = {"tanpa scaling": FunctionTransformer(), "Standard": StandardScaler(), "MinMax": MinMaxScaler(), "Robust": RobustScaler(),
           "Yeo-Johnson": PowerTransformer(), "Quantile(normal)": QuantileTransformer(n_quantiles=100, output_distribution="normal")}
models = {"logreg": LogisticRegression(max_iter=5000), "knn": KNeighborsClassifier(7), "svm_rbf": SVC(), "random_forest": RandomForestClassifier(200, random_state=SEED)}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
res = pd.DataFrame({m: {s: cross_val_score(make_pipeline(sc, mdl), X, y, cv=cv, scoring="f1_macro").mean() for s, sc in scalers.items()} for m, mdl in models.items()})
display(res.round(3))

Baca per kolom: model berbasis jarak/gradien (KNN, SVM, logreg) sangat terdampak scaling; **Random Forest hampir tidak** (pohon tidak peduli skala).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5)); im = ax.imshow(res.values, cmap="viridis", vmin=res.values.min()); ax.set_xticks(range(len(res.columns))); ax.set_xticklabels(res.columns, rotation=20)
ax.set_yticks(range(len(res.index))); ax.set_yticklabels(res.index)
for i in range(res.shape[0]):
    for j in range(res.shape[1]): ax.text(j, i, f"{res.values[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.savefig(OUT / "scaling.png", dpi=130); plt.show()
save_json({"f1_macro_cv": res.round(4).to_dict()}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Model pohon/boosting jarang butuh scaling atau pembersihan outlier; fokus di sana ke fitur dan validasi.
- Selalu ukur efek perlakuan pada validation yang tidak diperlakukan.